# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Do search-visible pages under-capture the clicks their position should predict, and can that
gap be scored and ranked into a reviewable action queue?

This supports a content team's weekly triage decision: which pages, out of hundreds, are
worth a human look this week. It does not attempt to explain WHY any single page
underperforms, or to predict future ranking changes.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Source: FlyRank Internship Warehouse Release (v20260703), pseudonymized, accessed via DuckDB
directly over hf:// — no bulk download.

Tables: dim_content (519,606 rows, content metadata) joined to
fact_content_daily_performance (78.8M rows, GSC daily metrics), filtered to month=2026-03 —
a mid-panel month, not the final-month _sample table, per this project's own leakage-safety
rule (the final month is reserved as a sealed test period).

After filtering to is_published=TRUE, is_deleted=FALSE, and gsc_data_available=TRUE, then
aggregating from page-day to page-month grain: 160,883 pages with real March 2026 search
signal.

Excluded: unpublished/deleted content; rows with no real GSC signal that day; any data
outside March 2026 (no forward-looking joins). No client names, domains, or raw queries
appear anywhere — all identifiers are dataset-issued pseudonymous hashes.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Step 1 — benchmark: built an empirical, impression-weighted expected-CTR curve by position
bucket (summed clicks ÷ summed impressions per bucket, not an average of noisy per-page
rates, since 60%+ of individual pages see zero clicks in any given window).

Step 2 — scoring: for each page, ctr_gap = expected_ctr(its bucket) − its actual ctr.
opportunity_score = ctr_gap × log1p(impressions) — impressions are log-dampened rather than
used raw, after an earlier version of this score was found to be dominated by a handful of
very-high-traffic pages from a few large clients.

Step 3 — reason codes: underperforming_for_position (real gap, ≥100 impressions),
gap_but_low_volume (gap exists but too little traffic to trust), or
meeting_or_beating_expected_ctr (no action indicated).

No baseline/model comparison in the precision@K sense here — this is a descriptive
benchmark-and-rank approach, not a trained classifier. A separate, parallel experiment
(decline prediction on momentum features) WAS validated with a client-grouped split and
found no generalizable signal — see w03_data_contract.ipynb's leakage trap and the earlier
capstone notebook's honest negative finding, both useful evidence this project takes
validation rigor seriously even where it produced a null result.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Of 160,883 scored pages: 68,803 (42.8%) flagged underperforming_for_position, 57,359
(35.7%) gap_but_low_volume, 34,721 (21.6%) meeting or beating expectation.

Median impressions among flagged underperforming pages: 718.

No formal baseline-vs-model table applies here (see Methodology) — the "result" is the
benchmark curve itself and the ranked list it produces, validated by the fact that flagged
pages show real, large CTR shortfalls relative to peers at the same position (not by a
precision@K score against ground truth, since no ground truth "should have reviewed this"
label exists in the data).

## 5. Limitations

*What this work cannot claim.*

- Position-bucket concentration: 69.6% of the top 1,000 flagged pages sit in position
  1-3 — a real consequence of using an absolute CTR gap (top positions have far higher
  expected CTR, so equal relative shortfalls produce larger absolute gaps there). This queue
  under-represents opportunities in positions 11+.
- One month, one snapshot (March 2026) — says nothing about whether a gap is new or
  long-standing.
- An unexplained anomaly in the benchmark itself: position 11-20 slightly out-clicks 6-10,
  on large enough samples that it isn't noise, but the cause is undetermined.
- Correlational only — a flagged page warrants review, not a diagnosis of cause (title,
  SERP feature, intent mismatch all remain untested explanations).
- No causal claims about search engine ranking behavior are made anywhere in this work.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

1. Review underperforming_for_position pages first, sorted by opportunity_score.
2. Treat gap_but_low_volume as a watchlist — recheck once more impressions accumulate.
3. Before using this for client-facing prioritization, be aware of the position-1-3
   concentration bias named in Limitations.
4. Investigate the 11-20/6-10 benchmark anomaly with a SERP-feature audit rather than
   assuming it's noise.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

- The CTR-by-position benchmark bar chart (already built as inline SVG in docs/index.html)
- Reason-code breakdown table (68,803 / 57,359 / 34,721) — source: work/outputs/action_playbook_summary.csv
- Top-20 ranked opportunity table — source: work/outputs/action_playbook_march2026.csv

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.


5-minute demo outline:
1. The question (30s) — do top-ranking pages always earn the clicks their position predicts?
2. The benchmark curve, live (1 min) — show the chart, name the 11-20/6-10 anomaly
3. The ranked queue + reason codes (2 min) — walk through 2-3 real flagged pages
4. The honest limitation (1 min) — position bias, why this isn't causal
5. What's next (30s) — fixing the concentration bias, testing the anomaly

Social post cut:
"Do page-1 rankings always earn the clicks they should? I benchmarked expected CTR by
search position across 160K+ pages and built a ranked opportunity list with reason codes —
including one honest surprise the data wouldn't let me smooth away. [link]"

3-sentence employer summary:
Built an empirical CTR-by-position benchmark and opportunity-scoring pipeline on FlyRank's
79M-row search warehouse, using DuckDB for at-scale querying without downloading data.
Flagged 68,803 pages with real, traffic-weighted underperformance and shipped them as a
ranked, reason-coded action queue with documented limitations. Caught and fixed a scoring
bias (impression-domination) before finalizing, and validated a separate model honestly
using a client-grouped split that revealed no leakage-free signal — reported as a negative
result rather than hidden.